In [34]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

In [35]:
df = pd.read_csv("data/real_vital_signs_dataset.csv") 
df.head()

,patient_id,timestamp,bpm,spo2,temperature,systolic_bp,diastolic_bp,respiratory_rate,context
0,P1064,2024-01-01 08:00:00,72,98.0,36.8,120,80,16,rest
1,P1064,2024-01-01 08:15:00,74,98.2,36.8,122,81,16,rest
2,P1064,2024-01-01 08:30:00,75,97.8,36.9,121,80,15,rest
3,P1064,2024-01-01 08:45:00,98,97.0,37.1,132,85,20,effort
4,P1064,2024-01-01 09:00:00,112,96.5,37.3,138,88,22,effort


In [36]:
def train_personalized_model(user_df):
    """
    Entraîne un modèle personnalisé sur l'ensemble des données d'un utilisateur
    en intégrant le niveau d'activité directement dans les caractéristiques (X).
    """
    # Conversion du contexte en niveau d'activité numérique (IMU)
    # 0 = repos, 1 = récupération, 2 = effort
    context_map = {'rest': 0, 'recovery': 1, 'effort': 2}
    user_df['activity_level'] = user_df['context'].map(context_map).fillna(0)
    
    # Matrice de caractéristiques complète
    features = ['bpm', 'spo2', 'temperature', 'activity_level']
    X_user = user_df[features]
    
    # Modèle d'anomalies multidimensionnel
    model = IsolationForest(contamination=0.05, random_state=42)
    model.fit(X_user)
    
    return model

In [37]:

def predict_anomaly(model, bpm, spo2, temperature, activity_level):
    """
    Évalue une nouvelle mesure. Renvoie uniquement le score du modèle ML.
    -1 = Anomalie (combinaison biométrique/activité inhabituelle)
     1 = Normal (conforme aux habitudes de cet utilisateur)
    """
    X_new = pd.DataFrame([[bpm, spo2, temperature, activity_level]], 
                         columns=['bpm', 'spo2', 'temperature', 'activity_level'])
    
    prediction = model.predict(X_new)[0]
    return prediction

In [38]:

p1064_history = df[df['patient_id'] == 'P1064'].copy()

model_p1064 = train_personalized_model(p1064_history)

# Test 1 : BPM à 112 pendant un effort (activity_level = 2)
res_effort = predict_anomaly(model_p1064, bpm=112, spo2=96.5, temperature=37.3, activity_level=2)
print("Effort normal (BPM 112 + Activity 2) :", res_effort) 
# Résultat attendu : 1 (Normal, car la combinaison BPM élevé + Mouvement est connue)

# Test 2 : BPM à 112 au repos (activity_level = 0)
res_anomalie = predict_anomaly(model_p1064, bpm=112, spo2=96.5, temperature=37.3, activity_level=0)
print("Anomalie au repos (BPM 112 + Activity 0) :", res_anomalie)
# Résultat attendu : -1 (Anomalie, car BPM élevé sans mouvement est incohérent)


Effort normal (BPM 112 + Activity 2) : -1
Anomalie au repos (BPM 112 + Activity 0) : -1
